## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the Digitalhub platform.

In [1]:
import digitalhub as dh
PROJECT_NAME = "cancer-classifier"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## 2. Execution

Fetch the "hello-world" operation in the project. 

In [4]:
func_mlflow_train_model = proj.get_function("cancer-classifier-train") 

Run the function as shown below. This will trigger the model training job on the platform and the model 'model-mlflow' get logged on to the platform.

In [5]:
func_mlflow_train_model.run(action='job')

{'kind': 'python+job:run', 'metadata': {'project': 'cancer-classifier', 'name': 'f1cae3bfa6b04577b3083cca91caaf6f', 'version': '00a4d85d0b6241d0841d9e3c38b8e4fb', 'created': '2026-04-14T14:54:56.965Z', 'updated': '2026-04-14T14:54:57.115Z', 'created_by': 'khurshid@fbk.eu', 'updated_by': 'khurshid@fbk.eu', 'relationships': [{'type': 'run_of', 'dest': 'store://cancer-classifier/function/python/cancer-classifier-train:81e409b8d1e24218b924bf370c67370f'}]}, 'spec': {'task': 'python+job://cancer-classifier/cbc8d24394c8492182d9ffd0714aa453', 'function': 'python://cancer-classifier/cancer-classifier-train:81e409b8d1e24218b924bf370c67370f', 'source': {'source': 'src/train.py', 'handler': 'breast_cancer_generator', 'base64': 'aW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4uZGF0YXNldHMgaW1wb3J0IGxvYWRfYnJlYXN0X2NhbmNlcgoKZnJvbSBkaWdpdGFsaHViX3J1bnRpbWVfcHl0aG9uIGltcG9ydCBoYW5kbGVyCgpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0Cgpmcm9tIHNrbGVhcm4uc3ZtIGltcG9ydCBTVkMKZnJvbSBwaW

Now, one can fetch the trained model saved in project.


In [8]:
model = proj.get_model("cancer_classifier")

Note: Alternatively, using the Core Management UI, one can navigate to 'Model' menu to see the generated model.

### Serve

The 'serve' action deploys MLflow ML models as services on Kubernetes. A Task is created by calling run() on the Function; task parameters are passed through that call.

In [15]:
serve_func = proj.new_function(
    name="cancer-classifier-service",
    kind="sklearnserve",
    model_name=model.name,
    path=model.key,
)

In [16]:
serve_run = serve_func.run("serve", wait=True)

2026-04-14 15:10:26,379 - digitalhub./home/khurshid/python3.10/lib/python3.10/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run 1d3ebb93956d433a87a74c1e6f39e045 to deploy service.
2026-04-14 15:10:31,395 - digitalhub./home/khurshid/python3.10/lib/python3.10/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run 1d3ebb93956d433a87a74c1e6f39e045 to deploy service.
2026-04-14 15:10:36,412 - digitalhub./home/khurshid/python3.10/lib/python3.10/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run 1d3ebb93956d433a87a74c1e6f39e045 to deploy service.
2026-04-14 15:10:36,428 - digitalhub./home/khurshid/python3.10/lib/python3.10/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Run 1d3ebb93956d433a87a74c1e6f39e045 service deployed.


#### Test the model

Let's test our deployed MLflow model by making a prediction request with sample Iris data:

In [17]:
import numpy as np

# Generate sample data for prediction
data = np.random.rand(2, 30).tolist()
json_payload = {
    "inputs": [{"name": "input-0", "shape": [2, 30], "datatype": "FP32", "data": data}]
}

# Make prediction
result = serve_run.refresh().invoke(model_name=model.name, json=json_payload).json()
print("Prediction result:")
print(result)

Prediction result:
{'model_name': 'cancer_classifier', 'id': '72ea7ae0-ecf0-42d3-96a2-497dc6449ae9', 'parameters': {}, 'outputs': [{'name': 'predict', 'shape': [2, 1], 'datatype': 'INT64', 'parameters': {'content_type': 'np'}, 'data': [1, 1]}]}
